# GroceryCo Audit - 05 - Tableau Data Prep & Dashboard Mockups

**Task 4.** Build the data Tableau will connect to, document the dashboard spec, and produce matplotlib mockups of each dashboard sheet so the final PDF deliverable shows what the dashboard will look like.

**Two outputs:**
1. `tableau_extract.csv` - one denormalised flat table containing every field Tableau needs (department, aisle, day-of-week, hour, reorder rate, items, p-values from t-tests). This is what you import into Tableau.
2. `dashboard_spec.md` - the full Tableau workbook spec (sheets, marks, calculated fields, filters, layout) you build from in ~30 minutes of Tableau work.

## Setup - auto-resolving paths

Run this cell first.

In [ ]:
# Auto-resolving path setup
from pathlib import Path

def find_project_root():
    p = Path.cwd().resolve()
    for parent in [p] + list(p.parents):
        if parent.name == "grocery_audit":
            return parent
        if (parent / "outputs").exists() and (parent / "data").exists() and (parent / "figures").exists():
            return parent
    return Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROJECT_ROOT = find_project_root()
DATASET_DIR  = PROJECT_ROOT.parent / "dataset"
DATA_DIR     = PROJECT_ROOT / "data"
OUTPUTS_DIR  = PROJECT_ROOT / "outputs"
FIGURES_DIR  = PROJECT_ROOT / "figures"

for d in [DATA_DIR, OUTPUTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

EXPECTED = ["aisles.csv", "departments.csv", "products.csv", "orders.csv", "order_products__train.csv"]
csv_paths = {}
for name in EXPECTED:
    for folder in [DATASET_DIR, DATA_DIR]:
        if (folder / name).exists():
            csv_paths[name] = folder / name
            break
missing = [f for f in EXPECTED if f not in csv_paths]
assert not missing, f"Missing CSVs: {missing}. Place them in {DATASET_DIR} or {DATA_DIR}"

print(f"Project root : {PROJECT_ROOT}")
print(f"Found {len(csv_paths)} CSVs")


## Step 1 - Build the denormalised Tableau extract

Tableau performs best with one wide flat file. We export the join of order_items × products × departments × aisles × orders, with computed columns Tableau will use directly (`is_weekend`, `time_window`).

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

conn = sqlite3.connect(OUTPUTS_DIR / "groceryco.db")

# One wide flat extract for Tableau
sql = '''
SELECT
    oi.order_id,
    oi.product_id,
    p.product_name,
    a.aisle,
    d.department,
    o.order_dow,
    o.order_hour_of_day,
    o.days_since_prior_order,
    oi.add_to_cart_order,
    oi.reordered,
    CASE WHEN o.order_dow IN (0, 6) THEN 'Weekend' ELSE 'Weekday' END AS day_segment,
    CASE
        WHEN o.order_hour_of_day < 6  THEN 'Night'
        WHEN o.order_hour_of_day < 12 THEN 'Morning'
        WHEN o.order_hour_of_day < 18 THEN 'Afternoon'
        ELSE 'Evening'
    END AS time_window,
    CASE
        WHEN o.days_since_prior_order BETWEEN 0 AND 3   THEN '01_0-3_days'
        WHEN o.days_since_prior_order BETWEEN 4 AND 7   THEN '02_4-7_days'
        WHEN o.days_since_prior_order BETWEEN 8 AND 14  THEN '03_8-14_days'
        WHEN o.days_since_prior_order BETWEEN 15 AND 21 THEN '04_15-21_days'
        WHEN o.days_since_prior_order BETWEEN 22 AND 29 THEN '05_22-29_days'
        WHEN o.days_since_prior_order = 30              THEN '06_30+_days_capped'
        ELSE NULL
    END AS lead_bucket
FROM order_items oi
JOIN products    p ON oi.product_id = p.product_id
JOIN aisles      a ON p.aisle_id = a.aisle_id
JOIN departments d ON p.department_id = d.department_id
JOIN orders      o ON oi.order_id = o.order_id
'''
print("Building flat extract from SQL... (1.38M rows)")
extract = pd.read_sql_query(sql, conn)
print(f"Extract built: {len(extract):,} rows x {len(extract.columns)} cols")
print(f"Memory: {extract.memory_usage(deep=True).sum() / 1024 / 1024:.1f} MB")
print(f"\nColumns: {list(extract.columns)}")
print(f"\nFirst 5 rows:")
print(extract.head().to_string(index=False))

# Save
extract_path = OUTPUTS_DIR / "tableau_extract.csv"
extract.to_csv(extract_path, index=False)
print(f"\nSaved: {extract_path}")
print(f"File size: {extract_path.stat().st_size / 1024 / 1024:.1f} MB")


## Step 2 - Build a small SUMMARY extract for fast Tableau prototyping

The full extract is 1.38M rows. For dashboard prototyping, a pre-aggregated department-by-day-by-window summary is much faster (a few hundred rows).

In [ ]:
summary = pd.read_sql_query('''
SELECT
    d.department,
    o.order_dow,
    CASE
        WHEN o.order_hour_of_day < 6  THEN 'Night'
        WHEN o.order_hour_of_day < 12 THEN 'Morning'
        WHEN o.order_hour_of_day < 18 THEN 'Afternoon'
        ELSE 'Evening'
    END AS time_window,
    COUNT(*) AS n_items,
    AVG(oi.reordered) AS reorder_rate
FROM order_items oi
JOIN products    p ON oi.product_id = p.product_id
JOIN departments d ON p.department_id = d.department_id
JOIN orders      o ON oi.order_id = o.order_id
GROUP BY d.department, o.order_dow, time_window
ORDER BY d.department, o.order_dow, time_window
''', conn)

# Add control limits computed in Notebook 03
spc = pd.read_csv(OUTPUTS_DIR / "spc_department_stats.csv")
grand_mean = spc["x_bar"].mean()
peer_std = spc["x_bar"].std()
ucl = grand_mean + 2 * peer_std
lcl = grand_mean - 2 * peer_std

summary["platform_mean"] = grand_mean
summary["UCL"]           = ucl
summary["LCL"]           = lcl
summary["status"] = summary["reorder_rate"].apply(
    lambda v: "OUT_HIGH" if v > ucl else "OUT_LOW" if v < lcl else "IN_CONTROL"
)
print(f"Summary: {len(summary):,} rows")
print(summary.head().to_string(index=False))

summary.to_csv(OUTPUTS_DIR / "tableau_summary_extract.csv", index=False)

conn.close()


## Step 3 - Document the Tableau dashboard spec

We write a Markdown spec describing every sheet in the workbook - what marks to use, what calculated fields to create, how to lay them out. This is the document you'll use to build the actual `.twbx` file in Tableau Desktop or Tableau Public.

In [ ]:
spec = '''# GroceryCo Operational Health Dashboard - Tableau Spec

## Data Source
- Connect to: `outputs/tableau_extract.csv` (full data, 1.38M rows)
- For prototyping: `outputs/tableau_summary_extract.csv` (~441 rows, fast)

## Calculated Fields (create in Tableau)

```
[Reorder Rate]            = AVG([Reordered])
[Items per Order]         = COUNT([Order Id]) / COUNTD([Order Id])
[Is Out of Control]       = IF [Reorder Rate] > [UCL] THEN "OUT_HIGH"
                            ELSEIF [Reorder Rate] < [LCL] THEN "OUT_LOW"
                            ELSE "IN_CONTROL" END
[Color OOC]               = CASE [Is Out of Control]
                              WHEN "OUT_HIGH" THEN "RED"
                              WHEN "OUT_LOW"  THEN "RED"
                              ELSE "GREEN" END
[Reorder Leakage]         = MAX(0, [Platform Mean] - [Reorder Rate])
[Leakage Volume]          = [Reorder Leakage] * SUM([N Items])
```

## Sheets

### Sheet 1 - X-bar Control Chart (Department Reorder Rates)
- **Marks**: Circle, sized by `n_items`, coloured by `Is Out of Control`
- **Columns**: `department` (sorted desc by `Reorder Rate`)
- **Rows**: `Reorder Rate` (continuous)
- **Reference Lines**: 3 horizontal lines at `[Platform Mean]`, `[UCL]`, `[LCL]`
  (use Analytics pane > Constant Line)
- **Tooltip**: department, reorder_rate, status, n_items, gap from platform mean

### Sheet 2 - Reorder Rate Heatmap (Day-of-Week x Hour)
- **Marks**: Square, coloured by `Reorder Rate` (red-yellow-green diverging at platform mean)
- **Columns**: `order_hour_of_day` (discrete, 0-23)
- **Rows**: `order_dow` (discrete, 0=Sun .. 6=Sat)
- **Tooltip**: dow, hour, reorder_rate, n_items

### Sheet 3 - Cascade Effect (Lead Time x Department)
- **Marks**: Square, coloured by `Reorder Rate`
- **Columns**: `lead_bucket` (sorted)
- **Rows**: `department` (sorted by reorder rate desc)
- **Tooltip**: department, lead_bucket, reorder_rate

### Sheet 4 - T-Test Results Card (table)
- Plain text or KPI cards showing the 3 t-test results from `outputs/ttest_results.csv`
- For each test: name, mean A, mean B, p-value, "REJECT H0" / "FAIL TO REJECT"
- Conditional format: green if p < 0.05, grey otherwise

### Sheet 5 - Reorder Leakage Report
- **Marks**: Bar, coloured by `[Color OOC]`
- **Rows**: `department` (filtered to status = OUT_LOW)
- **Columns**: `[Reorder Leakage]` (size of the gap below platform mean)
- **Tooltip**: department, current rate, platform mean, gap pp, t-test p-value
- This is the "Margin Leakage" analogue from the original brief.

### Sheet 6 - Filters & Parameters (global)
- Quick filters: `department`, `day_segment` (Weekend/Weekday), `time_window`
- Parameter: `Sigma Threshold` (default 3, slider 1-3)

## Dashboard Layout (1280 x 800)

```
+---------------------------------------------------------------+
| TITLE: GroceryCo Operational Health Dashboard                |
| Subtitle: Live monitoring with SPC and validated p-values     |
+---------------------------------------------------------------+
|                                                               |
|  Sheet 1 (Control Chart)        |  Sheet 4 (T-Tests)         |
|  ~60% width                     |  ~40% width                 |
|                                  |                             |
+---------------------------------+-----------------------------+
|                                                               |
|  Sheet 2 (DOW x Hour Heatmap)                                 |
|  Full width                                                   |
|                                                               |
+---------------------------------------------------------------+
|                                                               |
|  Sheet 3 (Cascade)              |  Sheet 5 (Leakage Report)  |
|  ~50% width                     |  ~50% width                 |
|                                                               |
+---------------------------------------------------------------+
|  Filters: [Department] [Day Segment] [Time Window]            |
+---------------------------------------------------------------+
```

## SQL Live Connection (alternative to CSV)

If using Tableau with a SQLite ODBC driver:
- DSN points to `outputs/groceryco.db`
- Use Custom SQL with the join from Step 1 of this notebook
- Performance: SQLite is fine up to ~10M rows; for production, port to PostgreSQL

## Refresh Strategy
- Extract (CSV): re-run notebook 05 to regenerate when data updates
- Live (SQLite): no refresh needed; queries hit the DB directly
'''
spec_path = OUTPUTS_DIR / "dashboard_spec.md"
spec_path.write_text(spec)
print(f"Spec saved: {spec_path}")
print(f"Length: {len(spec):,} chars")


## Step 4 - Build matplotlib mockups of each dashboard sheet

These are referenced by the final PDF (Notebook 06) so the deliverable shows what the dashboard looks like, even before you build the live `.twbx`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
sns.set_theme(style="whitegrid")

NAVY = "#1F3864"; TEAL = "#2E7D7D"; ORANGE = "#E07A1F"; GREEN = "#2E7D32"; RED = "#C00000"


### Mockup of the full dashboard (4-panel layout)

In [ ]:
fig = plt.figure(figsize=(18, 11))
fig.suptitle("GroceryCo Operational Health Dashboard - Mockup",
             fontsize=18, fontweight="bold", color=NAVY, y=0.985)

# Layout: 2x2 grid
gs = fig.add_gridspec(2, 2, hspace=0.40, wspace=0.20,
                       top=0.92, bottom=0.06, left=0.06, right=0.97)

# === PANEL 1 (top-left): X-bar control chart ===
ax1 = fig.add_subplot(gs[0, 0])
spc = pd.read_csv(OUTPUTS_DIR / "spc_department_stats.csv").sort_values("x_bar", ascending=False).reset_index(drop=True)
grand_mean = spc["x_bar"].mean()
peer_std = spc["x_bar"].std()
ucl, lcl = grand_mean + 2*peer_std, grand_mean - 2*peer_std
uwl, lwl = grand_mean + 1*peer_std, grand_mean - 1*peer_std

x = range(len(spc))
colors = [RED if s.startswith("OUT") else ORANGE if s.startswith("WARN") else GREEN
          for s in spc["status"]]
ax1.scatter(x, spc["x_bar"], s=80, c=colors, edgecolor="black", linewidth=1.0, zorder=5)
ax1.axhline(grand_mean, color=NAVY, linewidth=1.8, label=f"CL={grand_mean:.3f}")
ax1.axhline(ucl, color=RED, linewidth=1.2, linestyle="--", label=f"UCL/LCL")
ax1.axhline(lcl, color=RED, linewidth=1.2, linestyle="--")
ax1.fill_between(x, lcl, ucl, color=GREEN, alpha=0.08)
ax1.set_xticks(x)
ax1.set_xticklabels(spc["department"], rotation=70, ha="right", fontsize=8)
ax1.set_ylabel("Mean reorder rate")
ax1.set_title("Sheet 1 - X-bar Control Chart  (Department Reorder Rates)",
              fontweight="bold", fontsize=11, loc="left", color=NAVY)
ax1.legend(loc="upper right", fontsize=8)

# === PANEL 2 (top-right): T-test results card ===
ax2 = fig.add_subplot(gs[0, 1])
ax2.axis("off")
results = pd.read_csv(OUTPUTS_DIR / "ttest_results.csv")

ax2.text(0.0, 1.0, "Sheet 4 - T-Test Results", fontweight="bold",
         fontsize=11, transform=ax2.transAxes, color=NAVY, va="top")

y_pos = 0.88
for _, row in results.iterrows():
    sig = "REJECT H0" if row["significant"] else "FAIL TO REJECT"
    sig_color = GREEN if row["significant"] else "#888"
    ax2.text(0.0, y_pos, row["test"], fontsize=9.5, fontweight="bold",
             color=NAVY, transform=ax2.transAxes)
    ax2.text(0.0, y_pos - 0.05,
             f"  Mean A = {row['mean_a']:.4f}   Mean B = {row['mean_b']:.4f}",
             fontsize=8.5, color="#444", transform=ax2.transAxes,
             family="monospace")
    ax2.text(0.0, y_pos - 0.10,
             f"  t = {row['t_stat']:>+7.3f}   p = {row['p_value']:.3e}   d = {row['cohen_d']:+.3f}",
             fontsize=8.5, color="#444", transform=ax2.transAxes,
             family="monospace")
    ax2.text(0.0, y_pos - 0.15, f"  -> {sig}",
             fontsize=10, fontweight="bold", color=sig_color,
             transform=ax2.transAxes)
    y_pos -= 0.27

# === PANEL 3 (bottom-left): DOW x Hour heatmap ===
ax3 = fig.add_subplot(gs[1, 0])
import sqlite3
conn = sqlite3.connect(OUTPUTS_DIR / "groceryco.db")
heat_df = pd.read_sql_query('''
    SELECT o.order_dow, o.order_hour_of_day,
           AVG(oi.reordered) AS reorder_rate
    FROM orders o JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY o.order_dow, o.order_hour_of_day
''', conn)
conn.close()
heat = heat_df.pivot(index="order_dow", columns="order_hour_of_day", values="reorder_rate")
sns.heatmap(heat, cmap="RdYlGn", center=heat.values.mean(),
            cbar_kws={"label":"Reorder rate"}, ax=ax3,
            xticklabels=2, yticklabels=["Sun","Mon","Tue","Wed","Thu","Fri","Sat"])
ax3.set_title("Sheet 2 - Reorder Rate Heatmap  (DOW x Hour)",
              fontweight="bold", fontsize=11, loc="left", color=NAVY)
ax3.set_xlabel("Hour of day"); ax3.set_ylabel("")

# === PANEL 4 (bottom-right): Reorder Leakage Report ===
ax4 = fig.add_subplot(gs[1, 1])
spc_low = spc[spc["status"].isin(["OUT_LOW","WARN_LOW"])].sort_values("x_bar")
leakage = grand_mean - spc_low["x_bar"]
y = range(len(spc_low))
ax4.barh(y, leakage * 100, color=RED, edgecolor="black", linewidth=0.8)
ax4.set_yticks(y)
ax4.set_yticklabels(spc_low["department"], fontsize=9)
ax4.set_xlabel("Reorder rate gap below platform mean (percentage points)")
ax4.set_title("Sheet 5 - Reorder Leakage Report  (departments below LCL/LWL)",
              fontweight="bold", fontsize=11, loc="left", color=NAVY)
for i, (dept, gap) in enumerate(zip(spc_low["department"], leakage)):
    ax4.text(gap*100 + 0.3, i, f"-{gap*100:.1f}pp",
             va="center", fontsize=8, color="#444")

plt.savefig(FIGURES_DIR / "dashboard_mockup.png", dpi=150,
            bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {FIGURES_DIR / 'dashboard_mockup.png'}")


**Notebook 05 complete.** Tableau extract (`outputs/tableau_extract.csv`), summary (`outputs/tableau_summary_extract.csv`), spec (`outputs/dashboard_spec.md`), and dashboard mockup PNG all saved. Move to `06_audit_playbook.ipynb`.